<a href="https://colab.research.google.com/github/alexsouye09-lds/Motortriage/blob/main/app_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import classification_report, r2_score, mean_absolute_error

df = pd.read_csv("/content/startup_industrial_dataset.csv", encoding="utf-8")

EXCLURE = [
    "job_id", "motor_id", "shop_id", "intake_date", "data_origin",
    "generator_version", "suggested_temporal_split",
    "failure_mode_confirmed", "repair_quote_eur", "repair_quote_turnaround_days",
    "decision_final", "repair_scope_performed", "early_failure_12m",
]
X_COLS = [c for c in df.columns if c not in EXCLURE]
CAT_COLS = df[X_COLS].select_dtypes(exclude="number").columns.tolist()
X = df[X_COLS].copy()
X[CAT_COLS] = X[CAT_COLS].astype("category")

y_decision = (df["decision_final"] == "REPLACE").astype(int)
y_mode     = df["failure_mode_confirmed"]
y_quote    = np.log(df["repair_quote_eur"])
y_days     = df["repair_quote_turnaround_days"]


(X_tr, X_te,
 yd_tr, yd_te,
 ym_tr, ym_te,
 yq_tr, yq_te,
 yj_tr, yj_te) = train_test_split(
    X, y_decision, y_mode, y_quote, y_days,
    test_size=0.2, random_state=42, stratify=y_decision)


m_decision = HistGradientBoostingClassifier(random_state=0)
m_decision.fit(X_tr, yd_tr)
print(classification_report(yd_te, m_decision.predict(X_te)))


m_mode = HistGradientBoostingClassifier(random_state=0)
m_mode.fit(X_tr, ym_tr)
print(classification_report(ym_te, m_mode.predict(X_te)))

m_quote = HistGradientBoostingRegressor(random_state=0)
m_quote.fit(X_tr, yq_tr)
p = m_quote.predict(X_te)
print("R2", r2_score(yq_te, p), "MAE (log)", mean_absolute_error(yq_te, p))


m_days = HistGradientBoostingRegressor(random_state=0)
m_days.fit(X_tr, yj_tr)
print("MAE jours", mean_absolute_error(yj_te, m_days.predict(X_te)))

import pickle


categories = {c: list(X[c].cat.categories) for c in CAT_COLS}

modeles = {
    "m_decision.pkl": m_decision,
    "m_mode.pkl":     m_mode,
    "m_quote.pkl":    m_quote,
    "m_days.pkl":     m_days,
}

for nom_fichier, modele in modeles.items():
    paquet = {
        "model": modele,
        "x_cols": X_COLS,
        "categories": categories,
    }
    with open(nom_fichier, "wb") as f:
        pickle.dump(paquet, f)
    print("sauvegardé :", nom_fichier)

              precision    recall  f1-score   support

           0       0.78      0.79      0.79      5484
           1       0.82      0.81      0.82      6516

    accuracy                           0.80     12000
   macro avg       0.80      0.80      0.80     12000
weighted avg       0.80      0.80      0.80     12000

                             precision    recall  f1-score   support

            bearing_failure       0.95      0.97      0.96      4271
  contamination_or_moisture       0.86      0.83      0.85      1980
             no_fault_found       0.77      0.78      0.78       762
       rotor_bar_or_endring       0.91      0.87      0.89       652
shaft_or_housing_mechanical       0.96      0.91      0.93       685
  stator_insulation_failure       0.83      0.86      0.85      2199
          stator_turn_short       0.97      0.95      0.96      1451

                   accuracy                           0.90     12000
                  macro avg       0.89      0.88  

In [13]:
import pickle
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Triage moteurs", layout="wide")

# ---------- 1. Chargement des modèles (une seule fois) ----------
@st.cache_resource
def charger():
    noms = ["m_decision", "m_mode", "m_quote", "m_days"]
    return {n: pickle.load(open(f"{n}.pkl", "rb")) for n in noms}

P = charger()
x_cols = P["m_decision"]["x_cols"]
cats = P["m_decision"]["categories"]      # catégories valides, lues dans le .pkl

# ---------- 2. Formulaire ----------
st.title("Triage de moteurs : réparer ou remplacer ?")
st.warning("Modèles entraînés sur des données SYNTHÉTIQUES : démonstration, pas un outil de décision réel.")

saisie = {}
c1, c2, c3 = st.columns(3)

with c1:
    st.subheader("Moteur")
    saisie["rated_power_kw"] = st.number_input("Puissance (kW)", 0.75, 315.0, 11.0)
    saisie["pole_count"] = st.selectbox("Nombre de pôles", [2, 4, 6, 8], index=1)
    saisie["age_years"] = st.number_input("Âge (années)", 1, 50, 10)
    saisie["efficiency_class_nameplate"] = st.selectbox("Classe d'efficacité", cats["efficiency_class_nameplate"])
    saisie["vfd_driven"] = int(st.checkbox("Piloté par variateur"))

with c2:
    st.subheader("Contexte")
    saisie["application_type"] = st.selectbox("Application", cats["application_type"])
    saisie["environment"] = st.selectbox("Environnement", cats["environment"])
    saisie["customer_urgency"] = st.selectbox("Urgence", cats["customer_urgency"])
    saisie["reported_symptom"] = st.selectbox("Symptôme signalé", cats["reported_symptom"])
    saisie["new_motor_ie3_list_price_eur"] = st.number_input("Prix moteur neuf (€)", 0.0, 50000.0, 1000.0)

with c3:
    st.subheader("Mesures (optionnelles)")
    # Mesure non cochée = valeur manquante (NaN) : le modèle sait la gérer
    if st.checkbox("Résistance d'isolement mesurée"):
        saisie["insulation_resistance_1min_mohm"] = st.number_input("Isolement 1 min (MΩ)", 0.1, 20000.0, 500.0)
    if st.checkbox("Vibrations mesurées"):
        saisie["vibration_rms_mm_s"] = st.number_input("Vibration (mm/s RMS)", 0.1, 30.0, 1.5)
    if st.checkbox("Courant à vide mesuré"):
        saisie["no_load_current_pct_of_rated"] = st.number_input("Courant à vide (% nominal)", 10.0, 100.0, 35.0)

# ---------- 3. Construction de la ligne pour le modèle ----------
def construire_ligne(saisie):
    ligne = {c: np.nan for c in x_cols}   # tout ce qu'on ne connaît pas = NaN
    ligne.update(saisie)
    df = pd.DataFrame([ligne])[x_cols]    # même ordre de colonnes qu'à l'entraînement
    for c in x_cols:
        if c in cats:
            df[c] = pd.Categorical(df[c], categories=cats[c])
        else:
            df[c] = pd.to_numeric(df[c])
    return df

# ---------- 4. Prédiction ----------
if st.button("Analyser le moteur", type="primary"):
    ligne = construire_ligne(saisie)

    m_dec = P["m_decision"]["model"]
    p_remplacer = m_dec.predict_proba(ligne)[0][list(m_dec.classes_).index(1)]

    m_mode = P["m_mode"]["model"]
    proba_modes = pd.Series(m_mode.predict_proba(ligne)[0], index=m_mode.classes_).sort_values(ascending=False)

    devis = float(np.exp(P["m_quote"]["model"].predict(ligne)[0]))
    jours = float(P["m_days"]["model"].predict(ligne)[0])

    a, b, c = st.columns(3)
    a.metric("Probabilité de remplacement", f"{p_remplacer:.0%}")
    b.metric("Devis estimé", f"{devis:,.0f} €")
    c.metric("Délai estimé", f"{jours:.0f} jours")

    st.subheader("Cause probable de la panne")
    st.bar_chart(proba_modes.head(4))

2026-10-28 23:29:18.172 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 23:29:18.181 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 23:29:18.333 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 23:29:18.848 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-28 23:29:18.855 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 23:29:18.859 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-28 23:29:19.059 Thread 'MainThread'

In [12]:
!pip install streamlit scikit-learn pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [14]:
!pip install -q streamlit
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [15]:
!npm install -q localtunnel
!curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 4s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧34.11.218.167
⠙⠹your url is: https://tame-birds-hammer.loca.lt
^C


In [16]:
import subprocess, time, urllib.request

APP = r'''
import pickle
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Triage moteurs", layout="wide")

@st.cache_resource
def charger():
    noms = ["m_decision", "m_mode", "m_quote", "m_days"]
    return {n: pickle.load(open(f"/content/{n}.pkl", "rb")) for n in noms}

P = charger()
x_cols = P["m_decision"]["x_cols"]
cats = P["m_decision"]["categories"]

st.title("Triage de moteurs : réparer ou remplacer ?")
st.warning("Modèles entraînés sur des données SYNTHÉTIQUES : démonstration, pas un outil de décision réel.")

saisie = {}
c1, c2, c3 = st.columns(3)

with c1:
    st.subheader("Moteur")
    saisie["rated_power_kw"] = st.number_input("Puissance (kW)", 0.75, 315.0, 11.0)
    saisie["pole_count"] = st.selectbox("Nombre de pôles", [2, 4, 6, 8], index=1)
    saisie["age_years"] = st.number_input("Âge (années)", 1, 50, 10)
    saisie["efficiency_class_nameplate"] = st.selectbox("Classe d'efficacité", cats["efficiency_class_nameplate"])
    saisie["vfd_driven"] = int(st.checkbox("Piloté par variateur"))

with c2:
    st.subheader("Contexte")
    saisie["application_type"] = st.selectbox("Application", cats["application_type"])
    saisie["environment"] = st.selectbox("Environnement", cats["environment"])
    saisie["customer_urgency"] = st.selectbox("Urgence", cats["customer_urgency"])
    saisie["reported_symptom"] = st.selectbox("Symptôme signalé", cats["reported_symptom"])
    saisie["new_motor_ie3_list_price_eur"] = st.number_input("Prix moteur neuf (€)", 0.0, 50000.0, 1000.0)

with c3:
    st.subheader("Mesures (optionnelles)")
    if st.checkbox("Résistance d'isolement mesurée"):
        saisie["insulation_resistance_1min_mohm"] = st.number_input("Isolement 1 min (MΩ)", 0.1, 20000.0, 500.0)
    if st.checkbox("Vibrations mesurées"):
        saisie["vibration_rms_mm_s"] = st.number_input("Vibration (mm/s RMS)", 0.1, 30.0, 1.5)
    if st.checkbox("Courant à vide mesuré"):
        saisie["no_load_current_pct_of_rated"] = st.number_input("Courant à vide (% nominal)", 10.0, 100.0, 35.0)

def construire_ligne(saisie):
    ligne = {c: np.nan for c in x_cols}
    ligne.update(saisie)
    df = pd.DataFrame([ligne])[x_cols]
    for c in x_cols:
        if c in cats:
            df[c] = pd.Categorical(df[c], categories=cats[c])
        else:
            df[c] = pd.to_numeric(df[c])
    return df

if st.button("Analyser le moteur", type="primary"):
    ligne = construire_ligne(saisie)

    m_dec = P["m_decision"]["model"]
    p_remplacer = m_dec.predict_proba(ligne)[0][list(m_dec.classes_).index(1)]

    m_mode = P["m_mode"]["model"]
    proba_modes = pd.Series(m_mode.predict_proba(ligne)[0], index=m_mode.classes_).sort_values(ascending=False)

    devis = float(np.exp(P["m_quote"]["model"].predict(ligne)[0]))
    jours = float(P["m_days"]["model"].predict(ligne)[0])

    a, b, c = st.columns(3)
    a.metric("Probabilité de remplacement", f"{p_remplacer:.0%}")
    b.metric("Devis estimé", f"{devis:,.0f} €")
    c.metric("Délai estimé", f"{jours:.0f} jours")

    st.subheader("Cause probable de la panne")
    st.bar_chart(proba_modes.head(4))
'''

open("/content/app.py", "w").write(APP)

# Nettoyage d'anciens processus, puis installation
subprocess.run("pkill -f streamlit; pkill -f localtunnel", shell=True)
subprocess.run("pip install -q streamlit", shell=True)
subprocess.run("npm install -q localtunnel", shell=True)

# Lancement du serveur Streamlit en arrière-plan
subprocess.Popen("streamlit run /content/app.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1",
                 shell=True, cwd="/content")
time.sleep(8)

# Tunnel public
subprocess.Popen("npx localtunnel --port 8501 > /content/tunnel.txt 2>&1", shell=True)
time.sleep(8)

print("MOT DE PASSE du tunnel (IP) :", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
print(open("/content/tunnel.txt").read())

MOT DE PASSE du tunnel (IP) : 34.11.218.167
your url is: https://kind-breads-behave.loca.lt



In [18]:
%%writefile /content/app.py
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

BASE = Path(__file__).parent
CONTACT_EMAIL = "ton.email@exemple.com"   # <-- remplace par ton email

st.set_page_config(page_title="MotorTriage : réparer ou remplacer ?", page_icon="⚙️", layout="wide")

# ---------- Style ----------
st.markdown("""
<style>
.hero {background: linear-gradient(135deg,#0f2027 0%,#203a43 50%,#2c5364 100%);
       padding: 2.2rem 2rem; border-radius: 16px; color: white; margin-bottom: 1.2rem;}
.hero h1 {margin:0 0 .4rem 0; font-size: 2.2rem; color: white;}
.hero p {margin:0; font-size: 1.1rem; opacity:.92;}
.badge {display:inline-block; background:rgba(255,255,255,.15); padding:.25rem .7rem;
        border-radius:999px; font-size:.8rem; margin-bottom:.8rem;}
.step {background:#f4f7f9; border-radius:12px; padding:1rem; height:100%; color:#1b2a33;}
.step b {color:#2c5364;}
.verdict {padding:1.2rem 1.4rem; border-radius:14px; font-size:1.3rem; font-weight:700; margin:.6rem 0 1rem 0;}
.v-repair {background:#e6f6ec; color:#14532d; border-left:8px solid #22c55e;}
.v-replace {background:#fff1e6; color:#7c2d12; border-left:8px solid #f97316;}
.small {font-size:.85rem; opacity:.75;}
</style>
""", unsafe_allow_html=True)

# ---------- Modèles ----------
@st.cache_resource
def charger():
    noms = ["m_decision", "m_mode", "m_quote", "m_days"]
    return {n: pickle.load(open(BASE / f"{n}.pkl", "rb")) for n in noms}

P = charger()
x_cols = P["m_decision"]["x_cols"]
cats = P["m_decision"]["categories"]

# ---------- Libellés français ----------
FR = {
    "pump": "Pompe", "fan": "Ventilateur", "compressor": "Compresseur", "conveyor": "Convoyeur",
    "mixer_agitator": "Mélangeur / agitateur", "crusher_mill": "Broyeur / concasseur",
    "machine_tool": "Machine-outil", "other": "Autre",
    "clean_dry": "Propre et sec", "dusty": "Poussiéreux", "humid_washdown": "Humide / lavage",
    "outdoor_exposed": "Extérieur", "corrosive_chemical": "Corrosif / chimique",
    "standard": "Standard", "urgent": "Urgent", "critical_line_down": "Critique (ligne à l'arrêt)",
    "noise_or_vibration": "Bruit ou vibrations", "overheating": "Surchauffe",
    "tripping_or_no_start": "Disjonction / ne démarre pas", "seized_shaft": "Arbre bloqué",
    "high_current": "Courant élevé", "water_or_dirt_ingress": "Eau / saleté à l'intérieur",
    "none_planned_overhaul": "Aucun (révision prévue)",
}
CAUSES = {
    "bearing_failure": "Roulements", "stator_insulation_failure": "Isolation du bobinage",
    "stator_turn_short": "Court-circuit entre spires", "rotor_bar_or_endring": "Barres / anneau du rotor",
    "shaft_or_housing_mechanical": "Arbre / carter (mécanique)",
    "contamination_or_moisture": "Humidité / contamination", "no_fault_found": "Aucun défaut trouvé",
}
fr = lambda v: FR.get(v, v)

# ---------- En-tête marketing ----------
st.markdown("""
<div class="hero">
  <span class="badge">⚙️ Pour ateliers de réparation de moteurs électriques</span>
  <h1>Réparer ou remplacer ? Décidez en 30 secondes.</h1>
  <p>Saisissez les tests de réception : MotorTriage recommande, estime la cause de panne,
  le devis et le délai, avant même de démonter le moteur.</p>
</div>
""", unsafe_allow_html=True)

s1, s2, s3 = st.columns(3)
s1.markdown('<div class="step"><b>1. Saisir</b><br>Puissance, âge, contexte et quelques mesures de réception.</div>', unsafe_allow_html=True)
s2.markdown('<div class="step"><b>2. Analyser</b><br>Quatre modèles prédisent la décision, la cause, le devis et le délai.</div>', unsafe_allow_html=True)
s3.markdown('<div class="step"><b>3. Décider</b><br>Un devis cohérent et une recommandation claire à présenter au client.</div>', unsafe_allow_html=True)

st.write("")

# ---------- Formulaire ----------
saisie = {}
with st.form("form_moteur"):
    c1, c2, c3 = st.columns(3)
    with c1:
        st.subheader("🔧 Le moteur")
        saisie["rated_power_kw"] = st.number_input("Puissance (kW)", 0.75, 315.0, 11.0)
        saisie["pole_count"] = st.selectbox("Nombre de pôles", [2, 4, 6, 8], index=1)
        saisie["age_years"] = st.number_input("Âge (années)", 1, 50, 10)
        saisie["efficiency_class_nameplate"] = st.selectbox("Classe d'efficacité", cats["efficiency_class_nameplate"])
        saisie["vfd_driven"] = int(st.checkbox("Piloté par variateur de vitesse"))
    with c2:
        st.subheader("🏭 Le contexte")
        saisie["application_type"] = st.selectbox("Application", cats["application_type"], format_func=fr)
        saisie["environment"] = st.selectbox("Environnement", cats["environment"], format_func=fr)
        saisie["customer_urgency"] = st.selectbox("Urgence du client", cats["customer_urgency"], format_func=fr)
        saisie["reported_symptom"] = st.selectbox("Symptôme signalé", cats["reported_symptom"], format_func=fr)
        saisie["new_motor_ie3_list_price_eur"] = st.number_input("Prix d'un moteur neuf (€)", 0.0, 50000.0, 1000.0)
    with c3:
        st.subheader("📏 Mesures (optionnelles)")
        use_ir = st.checkbox("Résistance d'isolement mesurée")
        ir = st.number_input("Isolement 1 min (MΩ)", 0.1, 20000.0, 500.0)
        use_vib = st.checkbox("Vibrations mesurées")
        vib = st.number_input("Vibrations (mm/s RMS)", 0.1, 30.0, 1.5)
        use_nl = st.checkbox("Courant à vide mesuré")
        nl = st.number_input("Courant à vide (% du nominal)", 10.0, 100.0, 35.0)
    go = st.form_submit_button("Analyser le moteur", type="primary", use_container_width=True)

if use_ir: saisie["insulation_resistance_1min_mohm"] = ir
if use_vib: saisie["vibration_rms_mm_s"] = vib
if use_nl: saisie["no_load_current_pct_of_rated"] = nl

def construire_ligne(saisie):
    ligne = {c: np.nan for c in x_cols}   # non saisi = valeur manquante
    ligne.update(saisie)
    df = pd.DataFrame([ligne])[x_cols]
    for c in x_cols:
        if c in cats:
            df[c] = pd.Categorical(df[c], categories=cats[c])
        else:
            df[c] = pd.to_numeric(df[c])
    return df

# ---------- Résultats ----------
if go:
    ligne = construire_ligne(saisie)
    m_dec = P["m_decision"]["model"]
    p_rempl = float(m_dec.predict_proba(ligne)[0][list(m_dec.classes_).index(1)])
    m_mode = P["m_mode"]["model"]
    modes = pd.Series(m_mode.predict_proba(ligne)[0], index=m_mode.classes_).sort_values(ascending=False)
    devis = float(np.exp(P["m_quote"]["model"].predict(ligne)[0]))
    jours = float(P["m_days"]["model"].predict(ligne)[0])
    prix_neuf = saisie["new_motor_ie3_list_price_eur"]

    st.divider()
    if p_rempl >= 0.5:
        st.markdown(f'<div class="verdict v-replace">🔄 Remplacement recommandé ({p_rempl:.0%} de confiance)</div>', unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="verdict v-repair">🛠️ Réparation recommandée ({1-p_rempl:.0%} de confiance)</div>', unsafe_allow_html=True)

    a, b, c, d = st.columns(4)
    a.metric("Devis de réparation estimé", f"{devis:,.0f} €")
    b.metric("Délai estimé", f"{max(jours, 1):.0f} jours")
    if prix_neuf > 0:
        c.metric("Réparation / prix du neuf", f"{devis / prix_neuf:.0%}")
    d.metric("Cause la plus probable", CAUSES.get(modes.index[0], modes.index[0]))

    st.subheader("Causes de panne probables")
    for nom, p in modes.head(3).items():
        st.progress(min(max(float(p), 0.0), 1.0), text=f"{CAUSES.get(nom, nom)} : {p:.0%}")

    st.markdown('<p class="small">⚠️ Démonstration : modèles entraînés sur des données synthétiques, '
                'non validés sur de vrais moteurs. Ne pas utiliser comme outil de décision réel.</p>', unsafe_allow_html=True)

st.divider()
st.markdown(f"### 🤝 Vous dirigez un atelier de réparation ?\n"
            f"Nous cherchons des ateliers pilotes pour tester MotorTriage sur de vrais dossiers. "
            f"Écrivez-nous : **{CONTACT_EMAIL}**")

Overwriting /content/app.py


In [19]:
import subprocess, time, re, sklearn

open("/content/requirements.txt", "w").write(f"streamlit\nscikit-learn=={sklearn.__version__}\npandas\nnumpy\n")

subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
subprocess.run("pip install -q streamlit", shell=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True)

subprocess.Popen("streamlit run /content/app.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1", shell=True, cwd="/content")
time.sleep(8)
subprocess.Popen("/content/cloudflared tunnel --url http://localhost:8501 > /content/tunnel.txt 2>&1", shell=True)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.txt").read())
    if m:
        url = m.group(0); break
print("LIEN :", url or "pas encore prêt : relance la cellule ou lis /content/tunnel.txt")

LIEN : https://simpsons-plugin-displaying-departmental.trycloudflare.com


In [ ]:
import subprocess, time, re, sklearn

open("/content/requirements.txt", "w").write(f"streamlit\nscikit-learn=={sklearn.__version__}\npandas\nnumpy\n")

subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
subprocess.run("pip install -q streamlit", shell=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True)

subprocess.Popen("streamlit run /content/app.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1", shell=True, cwd="/content")
time.sleep(8)
subprocess.Popen("/content/cloudflared tunnel --url http://localhost:8501 > /content/tunnel.txt 2>&1", shell=True)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.txt").read())
    if m:
        url = m.group(0); break
print("LIEN :", url or "pas encore prêt : relance la cellule ou lis /content/tunnel.txt")

LIEN : https://simpsons-plugin-displaying-departmental.trycloudflare.com


In [20]:
import os
os.makedirs("/content/.streamlit", exist_ok=True)
open("/content/.streamlit/config.toml", "w").write(
    '[theme]\nbase = "light"\nprimaryColor = "#f97316"\nbackgroundColor = "#ffffff"\n'
    'secondaryBackgroundColor = "#f1f5f9"\ntextColor = "#0f172a"\n')

135

In [21]:
%%writefile /content/app.py
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

BASE = Path(__file__).parent
CONTACT_EMAIL = "ton.email@exemple.com"   # <-- remplace par ton email

st.set_page_config(page_title="MotorTriage : réparer ou remplacer ?", page_icon="⚙️", layout="wide")

# ---------- Style ----------
st.markdown("""
<style>
.block-container {padding-top: 1.5rem; max-width: 1150px;}
.hero {background: linear-gradient(120deg,#0b1f3a 0%,#0f4c81 55%,#13a89e 100%);
       padding: 2.8rem 2.4rem; border-radius: 22px; color: #fff; margin-bottom: 1.4rem;
       box-shadow: 0 12px 30px rgba(15,76,129,.28);}
.hero h1 {margin:0 0 .6rem 0; font-size: 2.7rem; line-height:1.1; color:#fff; font-weight:800;}
.hero h1 span {color:#ffb347;}
.hero p {margin:0 0 1.1rem 0; font-size: 1.15rem; opacity:.95; max-width: 760px;}
.pill {display:inline-block; background:rgba(255,255,255,.18); padding:.35rem .9rem; margin:0 .4rem .4rem 0;
       border-radius:999px; font-size:.85rem; font-weight:600; backdrop-filter: blur(4px);}
.feat {background:#fff; border-radius:16px; padding:1.1rem 1.2rem; height:100%;
       box-shadow:0 4px 14px rgba(15,23,42,.08); border-top:5px solid #13a89e; color:#0f172a;}
.feat.o {border-top-color:#f97316;} .feat.b {border-top-color:#0f4c81;} .feat.p {border-top-color:#8b5cf6;}
.feat .ic {font-size:1.8rem;} .feat h4 {margin:.3rem 0 .2rem 0; color:#0b1f3a;} .feat p {margin:0; font-size:.92rem; color:#475569;}
h2.sec {color:#0b1f3a; margin:1.6rem 0 .6rem 0; font-weight:800;}
[data-testid="stForm"] {background:#f8fafc; border:2px solid #e2e8f0; border-radius:18px; padding:1.4rem;}
[data-testid="stFormSubmitButton"] button {background: linear-gradient(90deg,#f97316,#ef4444); color:#fff; border:none;
       font-size:1.15rem; font-weight:700; padding:.8rem 1rem; border-radius:12px; box-shadow:0 8px 18px rgba(249,115,22,.35);}
[data-testid="stFormSubmitButton"] button:hover {filter:brightness(1.08); color:#fff;}
.verdict {padding:1.3rem 1.6rem; border-radius:16px; font-size:1.5rem; font-weight:800; margin:.8rem 0 1.1rem 0; color:#fff;}
.v-repair {background:linear-gradient(90deg,#059669,#10b981);} .v-replace {background:linear-gradient(90deg,#ea580c,#f97316);}
.kpi {background:#fff; border-radius:16px; padding:1rem 1.2rem; box-shadow:0 4px 14px rgba(15,23,42,.09); border-left:6px solid #0f4c81;}
.kpi.t {border-left-color:#13a89e;} .kpi.o {border-left-color:#f97316;} .kpi.p {border-left-color:#8b5cf6;}
.kpi .l {font-size:.8rem; text-transform:uppercase; letter-spacing:.05em; color:#64748b; font-weight:700;}
.kpi .v {font-size:1.7rem; font-weight:800; color:#0b1f3a; margin-top:.15rem;}
.barwrap {margin:.5rem 0 .9rem 0;} .barlab {display:flex; justify-content:space-between; font-weight:600; color:#0b1f3a; font-size:.95rem;}
.bar {background:#e2e8f0; border-radius:999px; height:14px; overflow:hidden; margin-top:.25rem;}
.bar div {height:100%; background:linear-gradient(90deg,#13a89e,#0f4c81); border-radius:999px;}
.cta {background:linear-gradient(120deg,#0b1f3a,#0f4c81); color:#fff; padding:2rem; border-radius:22px; text-align:center; margin-top:1.6rem;}
.cta h3 {color:#fff; margin:0 0 .5rem 0; font-size:1.7rem;} .cta p {opacity:.92; max-width:640px; margin:0 auto 1.1rem auto;}
.cta a {display:inline-block; background:linear-gradient(90deg,#f97316,#fbbf24); color:#0b1f3a !important; font-weight:800;
        padding:.85rem 1.6rem; border-radius:999px; text-decoration:none;}
.small {font-size:.82rem; color:#64748b;}
</style>
""", unsafe_allow_html=True)

# ---------- Modèles ----------
@st.cache_resource
def charger():
    noms = ["m_decision", "m_mode", "m_quote", "m_days"]
    return {n: pickle.load(open(BASE / f"{n}.pkl", "rb")) for n in noms}

P = charger()
x_cols = P["m_decision"]["x_cols"]
cats = P["m_decision"]["categories"]

# ---------- Libellés français ----------
FR = {
    "pump": "Pompe", "fan": "Ventilateur", "compressor": "Compresseur", "conveyor": "Convoyeur",
    "mixer_agitator": "Mélangeur / agitateur", "crusher_mill": "Broyeur / concasseur",
    "machine_tool": "Machine-outil", "other": "Autre",
    "clean_dry": "Propre et sec", "dusty": "Poussiéreux", "humid_washdown": "Humide / lavage",
    "outdoor_exposed": "Extérieur", "corrosive_chemical": "Corrosif / chimique",
    "standard": "Standard", "urgent": "Urgent", "critical_line_down": "Critique (ligne à l'arrêt)",
    "noise_or_vibration": "Bruit ou vibrations", "overheating": "Surchauffe",
    "tripping_or_no_start": "Disjonction / ne démarre pas", "seized_shaft": "Arbre bloqué",
    "high_current": "Courant élevé", "water_or_dirt_ingress": "Eau / saleté à l'intérieur",
    "none_planned_overhaul": "Aucun (révision prévue)",
}
CAUSES = {
    "bearing_failure": "Roulements", "stator_insulation_failure": "Isolation du bobinage",
    "stator_turn_short": "Court-circuit entre spires", "rotor_bar_or_endring": "Barres / anneau du rotor",
    "shaft_or_housing_mechanical": "Arbre / carter (mécanique)",
    "contamination_or_moisture": "Humidité / contamination", "no_fault_found": "Aucun défaut trouvé",
}
fr = lambda v: FR.get(v, v)

def kpi(label, value, cls=""):
    return f'<div class="kpi {cls}"><div class="l">{label}</div><div class="v">{value}</div></div>'

# ---------- Hero ----------
st.markdown("""
<div class="hero">
  <span class="pill">⚙️ Ateliers de réparation de moteurs électriques</span>
  <span class="pill">⚡ Résultat en quelques secondes</span>
  <h1>Réparer ou remplacer ?<br><span>Décidez avant de démonter.</span></h1>
  <p>Saisissez les tests de réception : MotorTriage recommande la meilleure option, estime la cause de panne,
  le devis et le délai, pour répondre à vos clients plus vite et avec plus d'assurance.</p>
</div>
""", unsafe_allow_html=True)

f1, f2, f3, f4 = st.columns(4)
f1.markdown('<div class="feat"><div class="ic">🎯</div><h4>Décision claire</h4><p>Réparer ou remplacer, avec un niveau de confiance.</p></div>', unsafe_allow_html=True)
f2.markdown('<div class="feat o"><div class="ic">💶</div><h4>Devis cohérent</h4><p>Une estimation de départ pour chiffrer plus vite.</p></div>', unsafe_allow_html=True)
f3.markdown('<div class="feat b"><div class="ic">🔍</div><h4>Cause probable</h4><p>Roulements, bobinage, rotor : où chercher en premier.</p></div>', unsafe_allow_html=True)
f4.markdown('<div class="feat p"><div class="ic">⏱️</div><h4>Délai annoncé</h4><p>Un délai estimé à communiquer au client.</p></div>', unsafe_allow_html=True)

st.markdown('<h2 class="sec">🔧 Analysez un moteur</h2>', unsafe_allow_html=True)

# ---------- Formulaire ----------
saisie = {}
with st.form("form_moteur"):
    c1, c2, c3 = st.columns(3)
    with c1:
        st.subheader("Le moteur")
        saisie["rated_power_kw"] = st.number_input("Puissance (kW)", 0.75, 315.0, 11.0)
        saisie["pole_count"] = st.selectbox("Nombre de pôles", [2, 4, 6, 8], index=1)
        saisie["age_years"] = st.number_input("Âge (années)", 1, 50, 10)
        saisie["efficiency_class_nameplate"] = st.selectbox(
            "Classe d'efficacité", cats["efficiency_class_nameplate"] + ["IE5"])
        saisie["vfd_driven"] = int(st.checkbox("Piloté par variateur de vitesse"))
    with c2:
        st.subheader("Le contexte")
        saisie["application_type"] = st.selectbox("Application", cats["application_type"], format_func=fr)
        saisie["environment"] = st.selectbox("Environnement", cats["environment"], format_func=fr)
        saisie["customer_urgency"] = st.selectbox("Urgence du client", cats["customer_urgency"], format_func=fr)
        saisie["reported_symptom"] = st.selectbox("Symptôme signalé", cats["reported_symptom"], format_func=fr)
        saisie["new_motor_ie3_list_price_eur"] = st.number_input("Prix d'un moteur neuf (€)", 0.0, 50000.0, 1000.0)
    with c3:
        st.subheader("Mesures (optionnelles)")
        use_ir = st.checkbox("Résistance d'isolement mesurée")
        ir = st.number_input("Isolement 1 min (MΩ)", 0.1, 20000.0, 500.0)
        use_vib = st.checkbox("Vibrations mesurées")
        vib = st.number_input("Vibrations (mm/s RMS)", 0.1, 30.0, 1.5)
        use_nl = st.checkbox("Courant à vide mesuré")
        nl = st.number_input("Courant à vide (% du nominal)", 10.0, 100.0, 35.0)
    go = st.form_submit_button("🚀 Analyser le moteur", type="primary", use_container_width=True)

if use_ir: saisie["insulation_resistance_1min_mohm"] = ir
if use_vib: saisie["vibration_rms_mm_s"] = vib
if use_nl: saisie["no_load_current_pct_of_rated"] = nl

ie5 = saisie["efficiency_class_nameplate"] == "IE5"
if ie5:
    saisie["efficiency_class_nameplate"] = "IE4"   # IE5 absent de l'entraînement : classe la plus proche

def construire_ligne(saisie):
    ligne = {c: np.nan for c in x_cols}   # non saisi = valeur manquante
    ligne.update(saisie)
    df = pd.DataFrame([ligne])[x_cols]
    for c in x_cols:
        if c in cats:
            df[c] = pd.Categorical(df[c], categories=cats[c])
        else:
            df[c] = pd.to_numeric(df[c])
    return df

# ---------- Résultats ----------
if go:
    ligne = construire_ligne(saisie)
    m_dec = P["m_decision"]["model"]
    p_rempl = float(m_dec.predict_proba(ligne)[0][list(m_dec.classes_).index(1)])
    m_mode = P["m_mode"]["model"]
    modes = pd.Series(m_mode.predict_proba(ligne)[0], index=m_mode.classes_).sort_values(ascending=False)
    devis = float(np.exp(P["m_quote"]["model"].predict(ligne)[0]))
    jours = float(P["m_days"]["model"].predict(ligne)[0])
    prix_neuf = saisie["new_motor_ie3_list_price_eur"]

    st.markdown('<h2 class="sec">📋 Résultat de l\'analyse</h2>', unsafe_allow_html=True)
    if ie5:
        st.info("IE5 n'existait pas dans les données d'entraînement : le modèle a utilisé IE4, la classe la plus proche.")
    if p_rempl >= 0.5:
        st.markdown(f'<div class="verdict v-replace">🔄 Remplacement recommandé · {p_rempl:.0%} de confiance</div>', unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="verdict v-repair">🛠️ Réparation recommandée · {1-p_rempl:.0%} de confiance</div>', unsafe_allow_html=True)

    k1, k2, k3, k4 = st.columns(4)
    k1.markdown(kpi("Devis estimé", f"{devis:,.0f} €", "o"), unsafe_allow_html=True)
    k2.markdown(kpi("Délai estimé", f"{max(jours, 1):.0f} jours", "t"), unsafe_allow_html=True)
    ratio = f"{devis / prix_neuf:.0%}" if prix_neuf > 0 else "n/a"
    k3.markdown(kpi("Réparation / prix du neuf", ratio, ""), unsafe_allow_html=True)
    k4.markdown(kpi("Cause probable", CAUSES.get(modes.index[0], modes.index[0]), "p"), unsafe_allow_html=True)

    st.markdown('<h2 class="sec">🔍 Causes de panne probables</h2>', unsafe_allow_html=True)
    for nom, p in modes.head(3).items():
        st.markdown(
            f'<div class="barwrap"><div class="barlab"><span>{CAUSES.get(nom, nom)}</span><span>{p:.0%}</span></div>'
            f'<div class="bar"><div style="width:{min(max(p,0),1)*100:.0f}%"></div></div></div>',
            unsafe_allow_html=True)

    st.markdown('<p class="small">⚠️ Démonstration : modèles entraînés sur des données synthétiques, '
                'non validés sur de vrais moteurs. Ne pas utiliser comme outil de décision réel.</p>', unsafe_allow_html=True)

# ---------- Appel à l'action ----------
st.markdown(f"""
<div class="cta">
  <h3>🤝 Vous dirigez un atelier de réparation ?</h3>
  <p>Nous cherchons des ateliers pilotes pour tester MotorTriage sur de vrais dossiers
  et l'adapter à votre façon de travailler.</p>
  <a href="mailto:{CONTACT_EMAIL}?subject=Atelier pilote MotorTriage">Devenir atelier pilote</a>
</div>
""", unsafe_allow_html=True)

Overwriting /content/app.py


In [24]:
%%bash
EMAIL="ouedaalexis@gmail.com"   # <-- remplace par ton email
PORT=8510

if ! ls /content/m_decision.pkl /content/m_mode.pkl /content/m_quote.pkl /content/m_days.pkl >/dev/null 2>&1; then
  echo "ERREUR : les 4 fichiers .pkl doivent etre dans /content"; exit 1
fi

mkdir -p /content/.streamlit
cat > /content/.streamlit/config.toml << 'EOF'
[theme]
base = "light"
primaryColor = "#f97316"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f1f5f9"
textColor = "#0f172a"
EOF

cat > /content/app.py << 'EOF'
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

BASE = Path(__file__).parent
CONTACT_EMAIL = "ton.email@exemple.com"   # <-- remplace par ton email

st.set_page_config(page_title="MotorTriage : réparer ou remplacer ?", page_icon="⚙️", layout="wide")

# ---------- Style ----------
st.markdown("""
<style>
.block-container {padding-top: 1.5rem; max-width: 1150px;}
.hero {background: linear-gradient(120deg,#0b1f3a 0%,#0f4c81 55%,#13a89e 100%);
       padding: 2.8rem 2.4rem; border-radius: 22px; color: #fff; margin-bottom: 1.4rem;
       box-shadow: 0 12px 30px rgba(15,76,129,.28);}
.hero h1 {margin:0 0 .6rem 0; font-size: 2.7rem; line-height:1.1; color:#fff; font-weight:800;}
.hero h1 span {color:#ffb347;}
.hero p {margin:0 0 1.1rem 0; font-size: 1.15rem; opacity:.95; max-width: 760px;}
.pill {display:inline-block; background:rgba(255,255,255,.18); padding:.35rem .9rem; margin:0 .4rem .4rem 0;
       border-radius:999px; font-size:.85rem; font-weight:600; backdrop-filter: blur(4px);}
.feat {background:#fff; border-radius:16px; padding:1.1rem 1.2rem; height:100%;
       box-shadow:0 4px 14px rgba(15,23,42,.08); border-top:5px solid #13a89e; color:#0f172a;}
.feat.o {border-top-color:#f97316;} .feat.b {border-top-color:#0f4c81;} .feat.p {border-top-color:#8b5cf6;}
.feat .ic {font-size:1.8rem;} .feat h4 {margin:.3rem 0 .2rem 0; color:#0b1f3a;} .feat p {margin:0; font-size:.92rem; color:#475569;}
h2.sec {color:#0b1f3a; margin:1.6rem 0 .6rem 0; font-weight:800;}
[data-testid="stForm"] {background:#f8fafc; border:2px solid #e2e8f0; border-radius:18px; padding:1.4rem;}
[data-testid="stFormSubmitButton"] button {background: linear-gradient(90deg,#f97316,#ef4444); color:#fff; border:none;
       font-size:1.15rem; font-weight:700; padding:.8rem 1rem; border-radius:12px; box-shadow:0 8px 18px rgba(249,115,22,.35);}
[data-testid="stFormSubmitButton"] button:hover {filter:brightness(1.08); color:#fff;}
.verdict {padding:1.3rem 1.6rem; border-radius:16px; font-size:1.5rem; font-weight:800; margin:.8rem 0 1.1rem 0; color:#fff;}
.v-repair {background:linear-gradient(90deg,#059669,#10b981);} .v-replace {background:linear-gradient(90deg,#ea580c,#f97316);}
.kpi {background:#fff; border-radius:16px; padding:1rem 1.2rem; box-shadow:0 4px 14px rgba(15,23,42,.09); border-left:6px solid #0f4c81;}
.kpi.t {border-left-color:#13a89e;} .kpi.o {border-left-color:#f97316;} .kpi.p {border-left-color:#8b5cf6;}
.kpi .l {font-size:.8rem; text-transform:uppercase; letter-spacing:.05em; color:#64748b; font-weight:700;}
.kpi .v {font-size:1.7rem; font-weight:800; color:#0b1f3a; margin-top:.15rem;}
.barwrap {margin:.5rem 0 .9rem 0;} .barlab {display:flex; justify-content:space-between; font-weight:600; color:#0b1f3a; font-size:.95rem;}
.bar {background:#e2e8f0; border-radius:999px; height:14px; overflow:hidden; margin-top:.25rem;}
.bar div {height:100%; background:linear-gradient(90deg,#13a89e,#0f4c81); border-radius:999px;}
.cta {background:linear-gradient(120deg,#0b1f3a,#0f4c81); color:#fff; padding:2rem; border-radius:22px; text-align:center; margin-top:1.6rem;}
.cta h3 {color:#fff; margin:0 0 .5rem 0; font-size:1.7rem;} .cta p {opacity:.92; max-width:640px; margin:0 auto 1.1rem auto;}
.cta a {display:inline-block; background:linear-gradient(90deg,#f97316,#fbbf24); color:#0b1f3a !important; font-weight:800;
        padding:.85rem 1.6rem; border-radius:999px; text-decoration:none;}
.small {font-size:.82rem; color:#64748b;}
</style>
""", unsafe_allow_html=True)

# ---------- Modèles ----------
@st.cache_resource
def charger():
    noms = ["m_decision", "m_mode", "m_quote", "m_days"]
    return {n: pickle.load(open(BASE / f"{n}.pkl", "rb")) for n in noms}

P = charger()
x_cols = P["m_decision"]["x_cols"]
cats = P["m_decision"]["categories"]

# ---------- Libellés français ----------
FR = {
    "pump": "Pompe", "fan": "Ventilateur", "compressor": "Compresseur", "conveyor": "Convoyeur",
    "mixer_agitator": "Mélangeur / agitateur", "crusher_mill": "Broyeur / concasseur",
    "machine_tool": "Machine-outil", "other": "Autre",
    "clean_dry": "Propre et sec", "dusty": "Poussiéreux", "humid_washdown": "Humide / lavage",
    "outdoor_exposed": "Extérieur", "corrosive_chemical": "Corrosif / chimique",
    "standard": "Standard", "urgent": "Urgent", "critical_line_down": "Critique (ligne à l'arrêt)",
    "noise_or_vibration": "Bruit ou vibrations", "overheating": "Surchauffe",
    "tripping_or_no_start": "Disjonction / ne démarre pas", "seized_shaft": "Arbre bloqué",
    "high_current": "Courant élevé", "water_or_dirt_ingress": "Eau / saleté à l'intérieur",
    "none_planned_overhaul": "Aucun (révision prévue)",
}
CAUSES = {
    "bearing_failure": "Roulements", "stator_insulation_failure": "Isolation du bobinage",
    "stator_turn_short": "Court-circuit entre spires", "rotor_bar_or_endring": "Barres / anneau du rotor",
    "shaft_or_housing_mechanical": "Arbre / carter (mécanique)",
    "contamination_or_moisture": "Humidité / contamination", "no_fault_found": "Aucun défaut trouvé",
}
fr = lambda v: FR.get(v, v)

def kpi(label, value, cls=""):
    return f'<div class="kpi {cls}"><div class="l">{label}</div><div class="v">{value}</div></div>'

# ---------- Hero ----------
st.markdown("""
<div class="hero">
  <span class="pill">⚙️ Ateliers de réparation de moteurs électriques</span>
  <span class="pill">⚡ Résultat en quelques secondes</span>
  <h1>Réparer ou remplacer ?<br><span>Décidez avant de démonter.</span></h1>
  <p>Saisissez les tests de réception : MotorTriage recommande la meilleure option, estime la cause de panne,
  le devis et le délai, pour répondre à vos clients plus vite et avec plus d'assurance.</p>
</div>
""", unsafe_allow_html=True)

f1, f2, f3, f4 = st.columns(4)
f1.markdown('<div class="feat"><div class="ic">🎯</div><h4>Décision claire</h4><p>Réparer ou remplacer, avec un niveau de confiance.</p></div>', unsafe_allow_html=True)
f2.markdown('<div class="feat o"><div class="ic">💶</div><h4>Devis cohérent</h4><p>Une estimation de départ pour chiffrer plus vite.</p></div>', unsafe_allow_html=True)
f3.markdown('<div class="feat b"><div class="ic">🔍</div><h4>Cause probable</h4><p>Roulements, bobinage, rotor : où chercher en premier.</p></div>', unsafe_allow_html=True)
f4.markdown('<div class="feat p"><div class="ic">⏱️</div><h4>Délai annoncé</h4><p>Un délai estimé à communiquer au client.</p></div>', unsafe_allow_html=True)

st.markdown('<h2 class="sec">🔧 Analysez un moteur</h2>', unsafe_allow_html=True)

# ---------- Formulaire ----------
saisie = {}
with st.form("form_moteur"):
    c1, c2, c3 = st.columns(3)
    with c1:
        st.subheader("Le moteur")
        saisie["rated_power_kw"] = st.number_input("Puissance (kW)", 0.75, 315.0, 11.0)
        saisie["pole_count"] = st.selectbox("Nombre de pôles", [2, 4, 6, 8], index=1)
        saisie["age_years"] = st.number_input("Âge (années)", 1, 50, 10)
        saisie["efficiency_class_nameplate"] = st.selectbox(
            "Classe d'efficacité", cats["efficiency_class_nameplate"] + ["IE5"])
        saisie["vfd_driven"] = int(st.checkbox("Piloté par variateur de vitesse"))
    with c2:
        st.subheader("Le contexte")
        saisie["application_type"] = st.selectbox("Application", cats["application_type"], format_func=fr)
        saisie["environment"] = st.selectbox("Environnement", cats["environment"], format_func=fr)
        saisie["customer_urgency"] = st.selectbox("Urgence du client", cats["customer_urgency"], format_func=fr)
        saisie["reported_symptom"] = st.selectbox("Symptôme signalé", cats["reported_symptom"], format_func=fr)
        saisie["new_motor_ie3_list_price_eur"] = st.number_input("Prix d'un moteur neuf (€)", 0.0, 50000.0, 1000.0)
    with c3:
        st.subheader("Mesures (optionnelles)")
        use_ir = st.checkbox("Résistance d'isolement mesurée")
        ir = st.number_input("Isolement 1 min (MΩ)", 0.1, 20000.0, 500.0)
        use_vib = st.checkbox("Vibrations mesurées")
        vib = st.number_input("Vibrations (mm/s RMS)", 0.1, 30.0, 1.5)
        use_nl = st.checkbox("Courant à vide mesuré")
        nl = st.number_input("Courant à vide (% du nominal)", 10.0, 100.0, 35.0)
    go = st.form_submit_button("🚀 Analyser le moteur", type="primary", use_container_width=True)

if use_ir: saisie["insulation_resistance_1min_mohm"] = ir
if use_vib: saisie["vibration_rms_mm_s"] = vib
if use_nl: saisie["no_load_current_pct_of_rated"] = nl

ie5 = saisie["efficiency_class_nameplate"] == "IE5"
if ie5:
    saisie["efficiency_class_nameplate"] = "IE4"   # IE5 absent de l'entraînement : classe la plus proche

def construire_ligne(saisie):
    ligne = {c: np.nan for c in x_cols}   # non saisi = valeur manquante
    ligne.update(saisie)
    df = pd.DataFrame([ligne])[x_cols]
    for c in x_cols:
        if c in cats:
            df[c] = pd.Categorical(df[c], categories=cats[c])
        else:
            df[c] = pd.to_numeric(df[c])
    return df

# ---------- Résultats ----------
if go:
    ligne = construire_ligne(saisie)
    m_dec = P["m_decision"]["model"]
    p_rempl = float(m_dec.predict_proba(ligne)[0][list(m_dec.classes_).index(1)])
    m_mode = P["m_mode"]["model"]
    modes = pd.Series(m_mode.predict_proba(ligne)[0], index=m_mode.classes_).sort_values(ascending=False)
    devis = float(np.exp(P["m_quote"]["model"].predict(ligne)[0]))
    jours = float(P["m_days"]["model"].predict(ligne)[0])
    prix_neuf = saisie["new_motor_ie3_list_price_eur"]

    st.markdown('<h2 class="sec">📋 Résultat de l\'analyse</h2>', unsafe_allow_html=True)
    if ie5:
        st.info("IE5 n'existait pas dans les données d'entraînement : le modèle a utilisé IE4, la classe la plus proche.")
    if p_rempl >= 0.5:
        st.markdown(f'<div class="verdict v-replace">🔄 Remplacement recommandé · {p_rempl:.0%} de confiance</div>', unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="verdict v-repair">🛠️ Réparation recommandée · {1-p_rempl:.0%} de confiance</div>', unsafe_allow_html=True)

    k1, k2, k3, k4 = st.columns(4)
    k1.markdown(kpi("Devis estimé", f"{devis:,.0f} €", "o"), unsafe_allow_html=True)
    k2.markdown(kpi("Délai estimé", f"{max(jours, 1):.0f} jours", "t"), unsafe_allow_html=True)
    ratio = f"{devis / prix_neuf:.0%}" if prix_neuf > 0 else "n/a"
    k3.markdown(kpi("Réparation / prix du neuf", ratio, ""), unsafe_allow_html=True)
    k4.markdown(kpi("Cause probable", CAUSES.get(modes.index[0], modes.index[0]), "p"), unsafe_allow_html=True)

    st.markdown('<h2 class="sec">🔍 Causes de panne probables</h2>', unsafe_allow_html=True)
    for nom, p in modes.head(3).items():
        st.markdown(
            f'<div class="barwrap"><div class="barlab"><span>{CAUSES.get(nom, nom)}</span><span>{p:.0%}</span></div>'
            f'<div class="bar"><div style="width:{min(max(p,0),1)*100:.0f}%"></div></div></div>',
            unsafe_allow_html=True)

    st.markdown('<p class="small">⚠️ Démonstration : modèles entraînés sur des données synthétiques, '
                'non validés sur de vrais moteurs. Ne pas utiliser comme outil de décision réel.</p>', unsafe_allow_html=True)

# ---------- Appel à l'action ----------
st.markdown(f"""
<div class="cta">
  <h3>🤝 Vous dirigez un atelier de réparation ?</h3>
  <p>Nous cherchons des ateliers pilotes pour tester MotorTriage sur de vrais dossiers
  et l'adapter à votre façon de travailler.</p>
  <a href="mailto:{CONTACT_EMAIL}?subject=Atelier pilote MotorTriage">Devenir atelier pilote</a>
</div>
""", unsafe_allow_html=True)
EOF

sed -i "s/ton.email@exemple.com/$EMAIL/" /content/app.py
SK=$(python -c "import sklearn; print(sklearn.__version__)")
printf "streamlit\nscikit-learn==$SK\npandas\nnumpy\n" > /content/requirements.txt

pkill -f "[s]treamlit run" 2>/dev/null
pkill -f "[c]loudflared" 2>/dev/null
sleep 2
pip install -q streamlit
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
chmod +x /content/cloudflared

cd /content
nohup streamlit run /content/app.py --server.port $PORT --server.headless true --browser.gatherUsageStats false > /content/logs.txt 2>&1 &
sleep 8
nohup /content/cloudflared tunnel --url http://localhost:$PORT > /content/tunnel.txt 2>&1 &

URL=""
for i in $(seq 1 30); do
  sleep 2
  URL=$(grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/tunnel.txt | head -1)
  if [ -n "$URL" ]; then break; fi
done
echo "=============================="
echo "LIEN : ${URL:-pas encore pret, relance la cellule ou lis /content/tunnel.txt}"
echo "=============================="

LIEN : https://flex-although-reception-tournament.trycloudflare.com


In [26]:
%%bash
cd /content
for f in m_decision m_mode m_quote m_days; do
  [ -f "$f.pkl" ] || { echo "MANQUANT : /content/$f.pkl"; exit 1; }
done

mkdir -p .streamlit
cat > .streamlit/config.toml << 'EOF'
[theme]
base = "light"
primaryColor = "#f97316"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f1f5f9"
textColor = "#0f172a"
EOF

python -c "import sklearn; print('streamlit\nscikit-learn==' + sklearn.__version__ + '\npandas\nnumpy')" > requirements.txt

cat > app.py << 'EOF'
import pickle
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import streamlit as st

BASE = Path(__file__).parent
CONTACT_EMAIL = "ouedaalexis@gmail.com"   # uniquement l'adresse, sans "mailto:"

st.set_page_config(page_title="MotorTriage", page_icon="⚙️", layout="wide")

# ---------- Style ----------
st.markdown("""
<style>
.block-container {padding-top: 1.5rem; max-width: 1150px;}
.hero {background: linear-gradient(120deg,#0b1f3a 0%,#0f4c81 55%,#13a89e 100%);
       padding: 2.8rem 2.4rem; border-radius: 22px; color: #fff; margin-bottom: 1.4rem;
       box-shadow: 0 12px 30px rgba(15,76,129,.28);}
.hero h1 {margin:0 0 .6rem 0; font-size: 2.7rem; line-height:1.1; color:#fff; font-weight:800;}
.hero h1 span {color:#ffb347;}
.hero p {margin:0 0 1.1rem 0; font-size: 1.15rem; opacity:.95; max-width: 760px;}
.pill {display:inline-block; background:rgba(255,255,255,.18); padding:.35rem .9rem; margin:0 .4rem .4rem 0;
       border-radius:999px; font-size:.85rem; font-weight:600; backdrop-filter: blur(4px);}
.feat {background:#fff; border-radius:16px; padding:1.1rem 1.2rem; height:100%;
       box-shadow:0 4px 14px rgba(15,23,42,.08); border-top:5px solid #13a89e; color:#0f172a;}
.feat.o {border-top-color:#f97316;} .feat.b {border-top-color:#0f4c81;} .feat.p {border-top-color:#8b5cf6;}
.feat .ic {font-size:1.8rem;} .feat h4 {margin:.3rem 0 .2rem 0; color:#0b1f3a;} .feat p {margin:0; font-size:.92rem; color:#475569;}
h2.sec {color:#0b1f3a; margin:1.6rem 0 .6rem 0; font-weight:800;}
[data-testid="stForm"] {background:#f8fafc; border:2px solid #e2e8f0; border-radius:18px; padding:1.4rem;}
[data-testid="stFormSubmitButton"] button {background: linear-gradient(90deg,#f97316,#ef4444); color:#fff; border:none;
       font-size:1.15rem; font-weight:700; padding:.8rem 1rem; border-radius:12px; box-shadow:0 8px 18px rgba(249,115,22,.35);}
[data-testid="stFormSubmitButton"] button:hover {filter:brightness(1.08); color:#fff;}
.verdict {padding:1.3rem 1.6rem; border-radius:16px; font-size:1.5rem; font-weight:800; margin:.8rem 0 1.1rem 0; color:#fff;}
.v-repair {background:linear-gradient(90deg,#059669,#10b981);} .v-replace {background:linear-gradient(90deg,#ea580c,#f97316);}
.kpi {background:#fff; border-radius:16px; padding:1rem 1.2rem; box-shadow:0 4px 14px rgba(15,23,42,.09); border-left:6px solid #0f4c81;}
.kpi.t {border-left-color:#13a89e;} .kpi.o {border-left-color:#f97316;} .kpi.p {border-left-color:#8b5cf6;}
.kpi .l {font-size:.8rem; text-transform:uppercase; letter-spacing:.05em; color:#64748b; font-weight:700;}
.kpi .v {font-size:1.7rem; font-weight:800; color:#0b1f3a; margin-top:.15rem;}
.barwrap {margin:.5rem 0 .9rem 0;} .barlab {display:flex; justify-content:space-between; font-weight:600; color:#0b1f3a; font-size:.95rem;}
.bar {background:#e2e8f0; border-radius:999px; height:14px; overflow:hidden; margin-top:.25rem;}
.bar div {height:100%; background:linear-gradient(90deg,#13a89e,#0f4c81); border-radius:999px;}
.cta {background:linear-gradient(120deg,#0b1f3a,#0f4c81); color:#fff; padding:2rem; border-radius:22px; text-align:center; margin-top:1.6rem;}
.cta h3 {color:#fff; margin:0 0 .5rem 0; font-size:1.7rem;} .cta p {opacity:.92; max-width:640px; margin:0 auto 1.1rem auto;}
.cta a {display:inline-block; background:linear-gradient(90deg,#f97316,#fbbf24); color:#0b1f3a !important; font-weight:800;
        padding:.85rem 1.6rem; border-radius:999px; text-decoration:none; margin:.25rem;}
.cta a.alt {background:#fff;}
.cta .addr {margin-top:.8rem; font-size:.95rem; opacity:.9;}
.small {font-size:.82rem; color:#64748b;}
</style>
""", unsafe_allow_html=True)

# ---------- Textes FR / DE ----------
TR = {
"fr": {
 "pill1": "⚙️ Ateliers de réparation de moteurs électriques", "pill2": "⚡ Résultat en quelques secondes",
 "h1a": "Réparer ou remplacer ?", "h1b": "Décidez avant de démonter.",
 "hero_p": "Saisissez les tests de réception : MotorTriage recommande la meilleure option, estime la cause de panne, le devis et le délai, pour répondre à vos clients plus vite et avec plus d'assurance.",
 "f1t": "Décision claire", "f1p": "Réparer ou remplacer, avec un niveau de confiance.",
 "f2t": "Devis cohérent", "f2p": "Une estimation de départ pour chiffrer plus vite.",
 "f3t": "Cause probable", "f3p": "Roulements, bobinage, rotor : où chercher en premier.",
 "f4t": "Délai annoncé", "f4p": "Un délai estimé à communiquer au client.",
 "sec_analyse": "🔧 Analysez un moteur",
 "motor": "Le moteur", "context": "Le contexte", "measures": "Mesures (optionnelles)",
 "power": "Puissance (kW)", "poles": "Nombre de pôles", "age": "Âge (années)", "effclass": "Classe d'efficacité",
 "vfd": "Piloté par variateur de vitesse", "application": "Application", "env": "Environnement",
 "urgency": "Urgence du client", "symptom": "Symptôme signalé", "newprice": "Prix d'un moteur neuf (€)",
 "use_ir": "Résistance d'isolement mesurée", "ir": "Isolement 1 min (MΩ)",
 "use_vib": "Vibrations mesurées", "vib": "Vibrations (mm/s RMS)",
 "use_nl": "Courant à vide mesuré", "nl": "Courant à vide (% du nominal)",
 "analyze": "🚀 Analyser le moteur", "result": "📋 Résultat de l'analyse",
 "ie5": "IE5 n'existait pas dans les données d'entraînement : le modèle a utilisé IE4, la classe la plus proche.",
 "replace": "🔄 Remplacement recommandé", "repair": "🛠️ Réparation recommandée", "conf": "de confiance",
 "k_quote": "Devis estimé", "k_days": "Délai estimé", "k_ratio": "Réparation / prix du neuf", "k_cause": "Cause probable",
 "days": "jours", "causes_title": "🔍 Causes de panne probables",
 "disclaimer": "⚠️ Démonstration : modèles entraînés sur des données synthétiques, non validés sur de vrais moteurs. Ne pas utiliser comme outil de décision réel.",
 "cta_title": "🤝 Vous dirigez un atelier de réparation ?",
 "cta_p": "Nous cherchons des ateliers pilotes pour tester MotorTriage sur de vrais dossiers et l'adapter à votre façon de travailler.",
 "cta_btn": "Devenir atelier pilote", "cta_gmail": "Ouvrir dans Gmail", "cta_addr": "Ou écrivez-nous :",
 "subject": "Atelier pilote MotorTriage",
},
"de": {
 "pill1": "⚙️ Werkstätten für die Reparatur von Elektromotoren", "pill2": "⚡ Ergebnis in Sekunden",
 "h1a": "Reparieren oder ersetzen?", "h1b": "Entscheiden Sie vor dem Zerlegen.",
 "hero_p": "Geben Sie die Eingangsprüfungen ein: MotorTriage empfiehlt die beste Option und schätzt Schadensursache, Angebotspreis und Bearbeitungszeit, damit Sie Ihren Kunden schneller und mit mehr Sicherheit antworten können.",
 "f1t": "Klare Entscheidung", "f1p": "Reparieren oder ersetzen, mit Konfidenzwert.",
 "f2t": "Konsistentes Angebot", "f2p": "Ein Richtwert, um schneller zu kalkulieren.",
 "f3t": "Wahrscheinliche Ursache", "f3p": "Lager, Wicklung, Rotor: wo man zuerst sucht.",
 "f4t": "Angekündigte Frist", "f4p": "Eine geschätzte Bearbeitungszeit für den Kunden.",
 "sec_analyse": "🔧 Motor analysieren",
 "motor": "Der Motor", "context": "Der Kontext", "measures": "Messwerte (optional)",
 "power": "Leistung (kW)", "poles": "Polzahl", "age": "Alter (Jahre)", "effclass": "Effizienzklasse",
 "vfd": "Mit Frequenzumrichter betrieben", "application": "Anwendung", "env": "Umgebung",
 "urgency": "Dringlichkeit des Kunden", "symptom": "Gemeldetes Symptom", "newprice": "Preis eines Neumotors (€)",
 "use_ir": "Isolationswiderstand gemessen", "ir": "Isolation 1 min (MΩ)",
 "use_vib": "Schwingungen gemessen", "vib": "Schwingungen (mm/s RMS)",
 "use_nl": "Leerlaufstrom gemessen", "nl": "Leerlaufstrom (% vom Nennwert)",
 "analyze": "🚀 Motor analysieren", "result": "📋 Analyseergebnis",
 "ie5": "IE5 war in den Trainingsdaten nicht enthalten: das Modell hat IE4 verwendet, die nächstliegende Klasse.",
 "replace": "🔄 Austausch empfohlen", "repair": "🛠️ Reparatur empfohlen", "conf": "Konfidenz",
 "k_quote": "Geschätztes Angebot", "k_days": "Geschätzte Bearbeitungszeit", "k_ratio": "Reparatur / Neupreis", "k_cause": "Wahrscheinliche Ursache",
 "days": "Tage", "causes_title": "🔍 Wahrscheinliche Schadensursachen",
 "disclaimer": "⚠️ Demo: Die Modelle wurden mit synthetischen Daten trainiert und nicht an echten Motoren validiert. Nicht als reales Entscheidungswerkzeug verwenden.",
 "cta_title": "🤝 Sie leiten eine Motorenwerkstatt?",
 "cta_p": "Wir suchen Pilotwerkstätten, die MotorTriage an echten Aufträgen testen und an Ihre Arbeitsweise anpassen.",
 "cta_btn": "Pilotwerkstatt werden", "cta_gmail": "In Gmail öffnen", "cta_addr": "Oder schreiben Sie uns:",
 "subject": "Pilotwerkstatt MotorTriage",
},
}

LBL = {
"fr": {
 "pump": "Pompe", "fan": "Ventilateur", "compressor": "Compresseur", "conveyor": "Convoyeur",
 "mixer_agitator": "Mélangeur / agitateur", "crusher_mill": "Broyeur / concasseur",
 "machine_tool": "Machine-outil", "other": "Autre",
 "clean_dry": "Propre et sec", "dusty": "Poussiéreux", "humid_washdown": "Humide / lavage",
 "outdoor_exposed": "Extérieur", "corrosive_chemical": "Corrosif / chimique",
 "standard": "Standard", "urgent": "Urgent", "critical_line_down": "Critique (ligne à l'arrêt)",
 "noise_or_vibration": "Bruit ou vibrations", "overheating": "Surchauffe",
 "tripping_or_no_start": "Disjonction / ne démarre pas", "seized_shaft": "Arbre bloqué",
 "high_current": "Courant élevé", "water_or_dirt_ingress": "Eau / saleté à l'intérieur",
 "none_planned_overhaul": "Aucun (révision prévue)",
},
"de": {
 "pump": "Pumpe", "fan": "Lüfter", "compressor": "Kompressor", "conveyor": "Förderband",
 "mixer_agitator": "Mischer / Rührwerk", "crusher_mill": "Brecher / Mühle",
 "machine_tool": "Werkzeugmaschine", "other": "Sonstige",
 "clean_dry": "Sauber und trocken", "dusty": "Staubig", "humid_washdown": "Feucht / Nassreinigung",
 "outdoor_exposed": "Außenbereich", "corrosive_chemical": "Korrosiv / chemisch",
 "standard": "Standard", "urgent": "Dringend", "critical_line_down": "Kritisch (Anlage steht)",
 "noise_or_vibration": "Geräusche oder Schwingungen", "overheating": "Überhitzung",
 "tripping_or_no_start": "Schutz löst aus / startet nicht", "seized_shaft": "Blockierte Welle",
 "high_current": "Erhöhter Strom", "water_or_dirt_ingress": "Wasser / Schmutz im Inneren",
 "none_planned_overhaul": "Keines (geplante Revision)",
},
}
CAUSES = {
"fr": {
 "bearing_failure": "Roulements", "stator_insulation_failure": "Isolation du bobinage",
 "stator_turn_short": "Court-circuit entre spires", "rotor_bar_or_endring": "Barres / anneau du rotor",
 "shaft_or_housing_mechanical": "Arbre / carter (mécanique)",
 "contamination_or_moisture": "Humidité / contamination", "no_fault_found": "Aucun défaut trouvé",
},
"de": {
 "bearing_failure": "Lager", "stator_insulation_failure": "Wicklungsisolation",
 "stator_turn_short": "Windungsschluss", "rotor_bar_or_endring": "Rotorstäbe / Kurzschlussring",
 "shaft_or_housing_mechanical": "Welle / Gehäuse (mechanisch)",
 "contamination_or_moisture": "Feuchtigkeit / Verunreinigung", "no_fault_found": "Kein Fehler gefunden",
},
}

# ---------- Choix de la langue ----------
choix = st.radio("🌐 Langue / Sprache", ["Français", "Deutsch"], horizontal=True, key="langue")
lang = "de" if choix == "Deutsch" else "fr"
t = lambda k: TR[lang][k]
fr = lambda v: LBL[lang].get(v, v)
cause = lambda v: CAUSES[lang].get(v, v)

def nombre(x, dec=0):
    s = f"{x:,.{dec}f}"
    return s.replace(",", " ").replace(".", ",") if lang == "fr" else s.replace(",", "X").replace(".", ",").replace("X", ".")

def pct(x):
    return f"{x:.0%}"

def kpi(label, value, cls=""):
    return f'<div class="kpi {cls}"><div class="l">{label}</div><div class="v">{value}</div></div>'

# ---------- Modèles ----------
@st.cache_resource
def charger():
    noms = ["m_decision", "m_mode", "m_quote", "m_days"]
    return {n: pickle.load(open(BASE / f"{n}.pkl", "rb")) for n in noms}

P = charger()
x_cols = P["m_decision"]["x_cols"]
cats = P["m_decision"]["categories"]

# ---------- Hero ----------
st.markdown(f"""
<div class="hero">
  <span class="pill">{t("pill1")}</span>
  <span class="pill">{t("pill2")}</span>
  <h1>{t("h1a")}<br><span>{t("h1b")}</span></h1>
  <p>{t("hero_p")}</p>
</div>
""", unsafe_allow_html=True)

f1, f2, f3, f4 = st.columns(4)
f1.markdown(f'<div class="feat"><div class="ic">🎯</div><h4>{t("f1t")}</h4><p>{t("f1p")}</p></div>', unsafe_allow_html=True)
f2.markdown(f'<div class="feat o"><div class="ic">💶</div><h4>{t("f2t")}</h4><p>{t("f2p")}</p></div>', unsafe_allow_html=True)
f3.markdown(f'<div class="feat b"><div class="ic">🔍</div><h4>{t("f3t")}</h4><p>{t("f3p")}</p></div>', unsafe_allow_html=True)
f4.markdown(f'<div class="feat p"><div class="ic">⏱️</div><h4>{t("f4t")}</h4><p>{t("f4p")}</p></div>', unsafe_allow_html=True)

st.markdown(f'<h2 class="sec">{t("sec_analyse")}</h2>', unsafe_allow_html=True)

# ---------- Formulaire (clés fixes : les valeurs restent quand on change de langue) ----------
saisie = {}
with st.form("form_moteur"):
    c1, c2, c3 = st.columns(3)
    with c1:
        st.subheader(t("motor"))
        saisie["rated_power_kw"] = st.number_input(t("power"), 0.75, 315.0, 11.0, key="w_power")
        saisie["pole_count"] = st.selectbox(t("poles"), [2, 4, 6, 8], index=1, key="w_poles")
        saisie["age_years"] = st.number_input(t("age"), 1, 50, 10, key="w_age")
        saisie["efficiency_class_nameplate"] = st.selectbox(
            t("effclass"), cats["efficiency_class_nameplate"] + ["IE5"], key="w_eff")
        saisie["vfd_driven"] = int(st.checkbox(t("vfd"), key="w_vfd"))
    with c2:
        st.subheader(t("context"))
        saisie["application_type"] = st.selectbox(t("application"), cats["application_type"], format_func=fr, key="w_app")
        saisie["environment"] = st.selectbox(t("env"), cats["environment"], format_func=fr, key="w_env")
        saisie["customer_urgency"] = st.selectbox(t("urgency"), cats["customer_urgency"], format_func=fr, key="w_urg")
        saisie["reported_symptom"] = st.selectbox(t("symptom"), cats["reported_symptom"], format_func=fr, key="w_sym")
        saisie["new_motor_ie3_list_price_eur"] = st.number_input(t("newprice"), 0.0, 50000.0, 1000.0, key="w_price")
    with c3:
        st.subheader(t("measures"))
        use_ir = st.checkbox(t("use_ir"), key="w_use_ir")
        ir = st.number_input(t("ir"), 0.1, 20000.0, 500.0, key="w_ir")
        use_vib = st.checkbox(t("use_vib"), key="w_use_vib")
        vib = st.number_input(t("vib"), 0.1, 30.0, 1.5, key="w_vib")
        use_nl = st.checkbox(t("use_nl"), key="w_use_nl")
        nl = st.number_input(t("nl"), 10.0, 100.0, 35.0, key="w_nl")
    go = st.form_submit_button(t("analyze"), type="primary", use_container_width=True)

def construire_ligne(saisie):
    ligne = {c: np.nan for c in x_cols}   # non saisi = valeur manquante
    ligne.update(saisie)
    df = pd.DataFrame([ligne])[x_cols]
    for c in x_cols:
        if c in cats:
            df[c] = pd.Categorical(df[c], categories=cats[c])
        else:
            df[c] = pd.to_numeric(df[c])
    return df

if go:
    if use_ir: saisie["insulation_resistance_1min_mohm"] = ir
    if use_vib: saisie["vibration_rms_mm_s"] = vib
    if use_nl: saisie["no_load_current_pct_of_rated"] = nl
    ie5 = saisie["efficiency_class_nameplate"] == "IE5"
    if ie5:
        saisie["efficiency_class_nameplate"] = "IE4"   # IE5 absent de l'entraînement
    st.session_state["derniere"] = {"saisie": dict(saisie), "ie5": ie5}

# ---------- Résultats (restent affichés quand on change de langue) ----------
if "derniere" in st.session_state:
    S = st.session_state["derniere"]["saisie"]
    ie5 = st.session_state["derniere"]["ie5"]
    ligne = construire_ligne(S)
    m_dec = P["m_decision"]["model"]
    p_rempl = float(m_dec.predict_proba(ligne)[0][list(m_dec.classes_).index(1)])
    m_mode = P["m_mode"]["model"]
    modes = pd.Series(m_mode.predict_proba(ligne)[0], index=m_mode.classes_).sort_values(ascending=False)
    devis = float(np.exp(P["m_quote"]["model"].predict(ligne)[0]))
    jours = float(P["m_days"]["model"].predict(ligne)[0])
    prix_neuf = S["new_motor_ie3_list_price_eur"]

    st.markdown(f'<h2 class="sec">{t("result")}</h2>', unsafe_allow_html=True)
    if ie5:
        st.info(t("ie5"))
    if p_rempl >= 0.5:
        st.markdown(f'<div class="verdict v-replace">{t("replace")} · {pct(p_rempl)} {t("conf")}</div>', unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="verdict v-repair">{t("repair")} · {pct(1-p_rempl)} {t("conf")}</div>', unsafe_allow_html=True)

    k1, k2, k3, k4 = st.columns(4)
    k1.markdown(kpi(t("k_quote"), f"{nombre(devis)} €", "o"), unsafe_allow_html=True)
    k2.markdown(kpi(t("k_days"), f"{nombre(max(jours, 1))} {t('days')}", "t"), unsafe_allow_html=True)
    ratio = pct(devis / prix_neuf) if prix_neuf > 0 else "n/a"
    k3.markdown(kpi(t("k_ratio"), ratio, ""), unsafe_allow_html=True)
    k4.markdown(kpi(t("k_cause"), cause(modes.index[0]), "p"), unsafe_allow_html=True)

    st.markdown(f'<h2 class="sec">{t("causes_title")}</h2>', unsafe_allow_html=True)
    for nom, p in modes.head(3).items():
        st.markdown(
            f'<div class="barwrap"><div class="barlab"><span>{cause(nom)}</span><span>{pct(p)}</span></div>'
            f'<div class="bar"><div style="width:{min(max(p,0),1)*100:.0f}%"></div></div></div>',
            unsafe_allow_html=True)

    st.markdown(f'<p class="small">{t("disclaimer")}</p>', unsafe_allow_html=True)

# ---------- Appel à l'action ----------
sujet = quote(t("subject"))
href_mail = f"mailto:{CONTACT_EMAIL}?subject={sujet}"
href_gmail = f"https://mail.google.com/mail/?view=cm&fs=1&to={CONTACT_EMAIL}&su={sujet}"
st.markdown(f"""
<div class="cta">
  <h3>{t("cta_title")}</h3>
  <p>{t("cta_p")}</p>
  <a href="{href_mail}">✉️ {t("cta_btn")}</a>
  <a class="alt" href="{href_gmail}" target="_blank" rel="noopener">{t("cta_gmail")}</a>
  <div class="addr">{t("cta_addr")} <b>{CONTACT_EMAIL}</b></div>
</div>
""", unsafe_allow_html=True)
EOF
python -m py_compile app.py && echo "app.py OK"

pkill -f "streamlit run" ; pkill -f cloudflared ; sleep 1
pip install -q streamlit
[ -x cloudflared ] || { wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared; }

nohup streamlit run app.py --server.port 8501 --server.headless true --browser.gatherUsageStats false > logs.txt 2>&1 &
sleep 8
nohup ./cloudflared tunnel --url http://localhost:8501 > tunnel.txt 2>&1 &

url=""
for i in $(seq 1 30); do
  url=$(grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.txt | head -1)
  [ -n "$url" ] && break
  sleep 2
done
echo "=============================="
echo "LIEN : ${url:-pas pret, relance la cellule (voir /content/tunnel.txt)}"
echo "=============================="

app.py OK
LIEN : https://plant-difference-begins-jesse.trycloudflare.com
